In [ ]:
import pandas as pd

from src.data_cleaning import *
from src.data_quality import *
from src.config import *

In [ ]:
# Loading raw data
applications = pd.read_csv("../data/raw/applications.csv")
daily = pd.read_csv("../data/raw/daily_surveys.csv")
final = pd.read_csv("../data/raw/final_survey.csv")

print("Applications:", applications.shape)
print("Daily:", daily.shape)
print("Final:", final.shape)


#### Data inspection

In [ ]:
applications.head()

In [ ]:
applications.info()

In [ ]:
applications["country"].value_counts()

#### Data quality checks

In [ ]:
app_report = validate_applications(applications)
daily_report = validate_daily(daily, applications)
final_report = validate_final(final, applications)

app_report.summary()
daily_report.summary()
final_report.summary()

#### Cleaning application dataset

In [ ]:
# standardise country names
applications = clean_countries(applications)

In [ ]:
#remove duplicates
applications = remove_duplicates(applications)

In [ ]:
#validate categories
applications["education_level"].value_counts()

In [ ]:
applications["education_level"] = applications["education_level"].str.lower()

#### cleaning daily survey dataset

In [ ]:
#checks missing values
daily.isnull().sum()
#drop missimg rows
daily = daily.dropna(subset=["student_id", "day", "engagement_score"])
#ensure valid ranges
daily = daily[
    (daily["engagement_score"] >= 1) &
    (daily["engagement_score"] <= 5)
]

#### Cleaning final survey dataset

In [ ]:
final = final.dropna(subset=["student_id", "completion"])
final["satisfaction"] = final["satisfaction"].clip(1, 5)
final["self_reported_learning"] = final["self_reported_learning"].clip(1, 5)

#### merge dataset

In [ ]:
# merge datasets

df = daily.merge(applications, on=["student_id", "year"], how="left")
df = df.merge(final, on=["student_id", "year"], how="left")

print("Merged shape:", df.shape)
df.head()

In [ ]:
# basic checks

print("Missing values after merge:\n")
print(df.isnull().sum())

In [ ]:
missing_students = df["student_id"].isnull().sum()
print("Missing student IDs after merge:", missing_students)

In [ ]:
# sanity checks after cleaning

df["country"].value_counts()

In [ ]:
df["disability_type"].value_counts()

In [ ]:
df["completion"].mean()

In [ ]:
#. save cleaned data
df.to_csv("../data/processed/merged_cleaned.csv", index=False)

In [ ]:
# run quality checks on cleaned data

app_report = validate_applications(applications)
app_report.summary()